In [13]:
import openai
from openai import OpenAI
import os
import io
import docx
import sys
import minsearch
import requests 
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import NMF
import torch
import json
import warnings
from elasticsearch import Elasticsearch
from tqdm.auto import tqdm

import gitsource


In [14]:
from dotenv import load_dotenv
load_dotenv("/workspace/.env")

False

# Data Ingestion

In [9]:
import requests
from minsearch import Index


def load_faq_data():
    docs_url = 'https://datatalks.club/faq/json/courses.json'
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = 'https://datatalks.club/faq'

    for course in courses_raw:
        course_url = f'{url_prefix}{course["path"]}'
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()

        documents.extend(course_data)

    return documents


def build_index(documents):
    index = Index(
        text_fields=['question', 'section', 'answer'],
        keyword_fields=['course']
    )
    index.fit(documents)
    return index

# RAG

In [10]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course='llm-zoomcamp',
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        boost_dict = {'question': 3.0, 'section': 0.5}
        filter_dict = {'course': self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['section'])
            lines.append('Q: ' + doc['question'])
            lines.append('A: ' + doc['answer'])
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer

# Call the RAG assistant

In [11]:
openai_client = OpenAI()

In [12]:
documents = load_faq_data()
index = build_index(documents)

assistant = RAGBase(index, openai_client)

In [13]:
assistant.rag('I just discovered the course, can I still join?')

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.'

# Function Calling and Agentic Loop

# Homework

First, we will pull the lesson pages straight from the course repository. 
We will use the commit `8c1834d` to make sure everyone works with the exact same data.

We will use `gitsource` for that:


In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

## Q1. How many lesson pages

How many lesson pages are in the dataset?

- 24
- **72**
- 240
- 720

In [6]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(f'{len(documents)} documentos parseados')
documents[0]

72 documentos parseados


{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

- `01-agentic-rag/lessons/03-rag.md`
- **`01-agentic-rag/lessons/14-agentic-loop.md`**
- `04-evaluation/lessons/13-llm-as-judge.md`
- `06-best-practices/lessons/02-hybrid-search.md`


In [8]:
from minsearch import Index
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)

In [ ]:
print("Top-1 filename:", results[0]['filename'])


for i, r in enumerate(results, 1):
    print(f"{i}. {r['filename']}")

Top-1 filename: 01-agentic-rag/lessons/14-agentic-loop.md
1. 01-agentic-rag/lessons/14-agentic-loop.md
2. 01-agentic-rag/lessons/15-frameworks.md
3. 01-agentic-rag/lessons/13-function-calling.md
4. 01-agentic-rag/lessons/11-agents-intro.md
5. 01-agentic-rag/lessons/16-other-frameworks.md



## Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper 
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?

- 700
- **7000**
- 70000
- 700000

We count input tokens instead of price because the cost depends on the model
and provider you use, but the size of the prompt we send is the same for
everyone.

Most LLM APIs report token usage on the response object (e.g.
`response.usage.input_tokens` / `prompt_tokens`). We'll read the input tokens
from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, `llm` returns only the text. Modify it to return the whole response, and change `rag` to return both the answer and usage (as a tuple or create a small dataclass for that).



In [15]:
openai_client = OpenAI()

In [16]:
class RAGLessons:
    def __init__(self, index, llm_client, model='gpt-5.4-mini'):
        self.index = index
        self.llm_client = llm_client
        self.model = model
        self.instructions = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''.strip()
        self.prompt_template = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

    # search simples — sem boost e sem filtro de curso
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    # contexto agora usa filename e content
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"FILE: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(question=query, context=context)

    # llm agora devolve o RESPONSE inteiro (pra ter acesso ao usage)
    def llm(self, prompt):
        return self.llm_client.responses.create(
            model=self.model,
            input=[
                {'role': 'developer', 'content': self.instructions},
                {'role': 'user', 'content': prompt},
            ],
        )

    # rag devolve (answer, usage) — pra você ler os tokens
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response.usage


In [17]:
assistant = RAGLessons(index, openai_client)

query = "How does the agentic loop keep calling the model until it stops?"
answer, usage = assistant.rag(query)

print("ANSWER:")
print(answer)
print()
print("USAGE:")
print(usage)
print(f"\ninput_tokens: {usage.input_tokens}")


ANSWER:
The loop keeps calling the model with a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there are function calls, the code runs the tools, appends the tool outputs to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop.

So the stop condition is: **no function calls in the latest response**.

USAGE:
ResponseUsage(input_tokens=7125, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=92, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7217)

input_tokens: 7125



## Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

- 70
- **295**
- 1100
- 4500


In [18]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f'{len(chunks)} chunks')
chunks[0]


295 chunks


{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon


## Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?

- about the same
- **3× fewer**
- 10× fewer
- 30× fewer



In [19]:
# Mesmo padrão da Q2, mas indexando CHUNKS em vez de documents
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunk_index.fit(chunks)

# Mesmo RAGLessons da Q3, agora apontando pro índice de chunks
assistant_chunks = RAGLessons(chunk_index, openai_client)

query = "How does the agentic loop keep calling the model until it stops?"
answer, usage = assistant_chunks.rag(query)

print("ANSWER:")
print(answer)
print()
print(f"input_tokens (chunked): {usage.input_tokens}")


ANSWER:
The loop keeps calling the model inside a `while True` loop. Each turn it checks whether the model returned any `function_call` items:

- If there are function calls, it runs them, adds the results to `messages`, and continues.
- If there are no function calls, it breaks out of the loop.

So the stopping condition is:

```python
if has_function_calls == False:
    break
```

In short: it keeps looping until the model returns a final message with no tool calls.

input_tokens (chunked): 2308


In [20]:
_, usage_q3 = assistant.rag(query)         # da Q3 (índice de documents)
_, usage_q5 = assistant_chunks.rag(query)  # da Q5 (índice de chunks)

print(f"Q3 (sem chunking): {usage_q3.input_tokens} tokens")
print(f"Q5 (com chunking): {usage_q5.input_tokens} tokens")
print(f"Razão: {usage_q3.input_tokens / usage_q5.input_tokens:.1f}× menos")


Q3 (sem chunking): 7125 tokens
Q5 (com chunking): 2308 tokens
Razão: 3.1× menos



## Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

```bash
uv add toyaikit
```

Create a `search` function that uses the chunk index. Give it a type hint and
a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your `search` tool and run it (with toyaikit, the same way
as in the ToyAIKit lesson). Use these instructions for the agent (they nudge
it to search a few times):

> You're a course teaching assistant. Answer the student's question using the
> search tool. Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many
times it called the `search` tool.

How many times did the agent call `search`?

> Note: the agent decides this itself, so it varies a little between runs -
> pick the closest option. We measured this with OpenAI `gpt-5.4-mini`; with a
> different model or provider the number may differ, so keep that in mind.

* 0
* 4
* 10
* 20

In [21]:
from toyaikit.chat import ChatAssistant, IPythonChatInterface, OpenAIClient
from toyaikit.tools import Tools

# Contador pra responder a pergunta da homework
search_count = 0

def search(query: str) -> list:
    """Search the course lesson chunks for content matching the query.

    Args:
        query: The search query string.

    Returns:
        A list of chunks (each with filename and content) ordered by relevance.
    """
    global search_count
    search_count += 1
    return chunk_index.search(query, num_results=5)

# Registra o search como tool
tools = Tools()
tools.add_tool(search)

# Configura cliente e interface
llm_client = OpenAIClient(model='gpt-5.4-mini', client=openai_client)
chat_interface = IPythonChatInterface()

developer_prompt = """You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering."""

agent = ChatAssistant(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    llm_client=llm_client,
)

agent.run()   # ABRE o chat interativo no notebook


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.
